In [31]:
# 参考　https://qiita.com/rairaii/items/8673b117096eb0e7e267

from typesafe_sdk import Choice, Noul, Score, TypeSafeClient

client = TypeSafeClient()

ticket = "お世話になっております。3日前からStripeアカウントを連携しようとしているのですが、何度やっても失敗します。売上に影響が出ていますので、至急対応をお願いします。"

# 非常に長いクエリでもレスポンス速度や精度が落ちないかを検証するため、長文のチケットを入力として与える
# ticket = """
# お世話になっております。株式会社〇〇のシステム管理者です。
# 現在、貴社サービスと弊社のStripe連携において極めて重大かつ緊急のシステム障害が発生しており、大変困惑し、憤りを感じております。

# 【障害の発生経緯と状況】
# 先週の木曜日（約6日前）より、弊社ECサイトにて顧客がStripe経由で決済を完了しようとすると「エラーコード: ERR_STRIPE_OAUTH_REFUSED」が頻発するようになりました。当初は一時的なネットワークエラーかと静観しておりましたが、昨日からはすべてのStripe連携決済が完全に応答不可となり、エラー率が100%に達しております。

# 【ビジネス上の影響と被害額】
# 弊社は現在、年内で最も売上が集中する大型セール期間中であり、この決済障害により過去3日間で概算して約1,200万円以上の機会損失（決済失敗による離脱）が発生しています。すでに数十名の顧客から「二重課金になっているのではないか」「決済画面から進まない」といった苦情のお問い合わせが殺到しており、弊社のサポートデスクもパンク状態です。

# 【これまでの対応と貴社サポートへの不満】
# すでに一昨日の時点で一度貴社のお問い合わせフォームより一次報告を送信いたしましたが、48時間以上経過した現在も自動返信メール以外の回答が一切届いておりません。これほどの重大障害に対して放置されている状況は、事業者として到底納得がいきません。また、ドキュメントに記載されているOAuth再連携コマンドもすべて試しましたが、状況は一向に改善されません。

# 【要求事項】
# 1. 貴社技術チームによる本件Stripe連携障害の緊急調査および本日中の復旧対応。
# 2. 過去3日間で不整合が発生している決済ログ（約350件分）のエラー原因と、二重課金有無の個別レポート提出。
# 3. 過去に発生した決済失敗に対する返金処理の手順、および今回生じた機会損失に対する損害補償または月額利用料の免除についての見解提示。

# 本日17:00までに技術責任者様またはアカウントマネージャー様よりお電話（090-XXXX-XXXX）または本メール宛てに直接のご連絡をいただけない場合、大変遺憾ではございますが、顧問弁護士と相談の上、利用契約の即時解約および法的措置を含めた対応を検討せざるを得ません。

# 極めて急を要する事態ですので、本メッセージを確認され次第、最優先でご対応をお願いいたします。
# """

response = client.system_one(
    state=ticket,
    questions={
        "department": Choice(   # Choiceは、複数の選択肢の中から最も適切なものを選ぶための質問タイプです
            instructions="どの部署が対応すべきか",
            criteria={
                "billing": "支払いやサブスクリプションに関する問題",
                "technical": "不具合や連携に関する問題",
                "sales": "料金やアカウントに関する質問",
            },
        ),
        "frustration": Score(  # Scoreは、0.0〜(n-1).0（連続的なインデックスの値域。今回は選択肢が３つなので0〜2.0）の範囲で数値を返す質問タイプです
            instructions="顧客がどの程度苛立っているか",
            criteria=[
                "冷静で、事実を述べているだけ",
                "苛立っているが、丁寧さは保っている",
                "非常に怒っており、強い言葉を使っている",
            ],
        ),
        "is_urgent": Noul(  # Noulは、0.0〜1.0の範囲で数値を返す質問タイプです
            instructions="メッセージから緊急性や時間的な切迫感が伝わる",
        ),
    },
)

print(response.answers["department"].choice)  # "technical"
print(response.answers["frustration"].score)  # 0.96〜0.97
print(response.answers["is_urgent"].noul)     # 0.97


technical
0.96
0.97


In [32]:
## 1. 感情の微細なグラデーションテスト（`Score` の解像度検証）

# テキストに含まれる「不満」と「期待」の絶妙なバランスをスコアリングさせ、境界値の判定精度を検証します。

# * **検証目的:** `Score` が文脈の細かなニュアンス（パッシブアグレッシブ、遠回しな嫌みなど）を数値に正しく反映できるか。
# * **テストケース（入力例）:**

review_text = "「新機能の追加ありがとうございます。使いこなすにはかなり熟練が必要そうですが、マニュアルを解読しながらなんとか対応してみます。次回のアプデにも期待していますね。」"

response = client.system_one(
    state=review_text,
    questions={
        "satisfaction_level": Score(
            instructions="ユーザーの製品に対する本当の満足度",
            criteria=[
                "非常に不満（遠回しな批判や皮肉を含む）",
                "やや不満・困惑（使いづらさを感じている）",
                "中立・どちらでもない",
                "概ね満足（肯定的に受け止めている）",
                "非常に満足（絶賛している）",
            ],
        ),
    },
)

print(response.answers["satisfaction_level"].score)  # 0.96〜0.97


## 2. 相反する要素の優先度判定（`Choice` の頑健性検証）

# 1つのドキュメントに「返金要求」「バグ報告」「営業の相談」が同時に含まれている場合に、最も主要なタスクを正しく識別できるかをテストします。

# * **検証目的:** 複数のトピックが混在したノイズの多い文章でも、主要なカテゴリを1つに絞り込めるか。
# * **テストケース（入力例）:**

multi_topic_text = "「来期のエンタープライズプラン導入に向けて商談を進めたいのですが、一点問題があります。現在お試し中のアカウントでCSVエクスポートを行うと500エラーになり、検証がストップしています。また、先月誤って決済されたテストカードの払い戻しも併せて確認したいです。」"

response = client.system_one(
    state=multi_topic_text,
    questions={
        "primary_intent": Choice(
            instructions="この問い合わせで最も優先して対応すべき目的",
            criteria={
                "sales_opportunity": "新規契約や商談に関する機会",
                "technical_bug": "システムの不具合やエラーの修正",
                "billing_refund": "返金や過去の請求に関する手続き",
            },
        ),
    },
)

print(response.answers["primary_intent"].choice)  # "technical"

## 3. 潜在的リスク・裏の文脈の検知（`Noul` の確率精度検証）

# 「解約したい」とは一言も書かれていない文章から、顧客の離脱リスク（チャーンリスク）や法的リスクを正しく検知できるか判定します。

# * **検証目的:** 表面上の単語に惑わされず、潜在的な意思や状況の切迫感を二値/確率として出力できるか。
# * **テストケース（入力例）:**
inquiry_text = "「最近、競合の〇〇社さんが似たような機能を半額でリリースされたのを見かけました。弊社でもコスト見直しが入っておりまして、現在の契約更新時期（来月）までに社内で他社ツールとの比較検討プレゼンを行う予定です。つきましては、現在の利用実績データをAPI等で一括抽出する方法を教えてください。」"

response = client.system_one(
    state=inquiry_text,
    questions={
        "is_churn_risk": Noul(
            instructions="この顧客が近いうちに解約または他社へ乗り換える可能性が高い",
        ),
        "is_legal_risk": Noul(
            instructions="損害賠償や弁護士の介入など法的トラブルに発展する予兆がある",
        ),
    },
)

print(response.answers["is_churn_risk"].noul)       # 0.97
print(response.answers["is_legal_risk"].noul)       # 0.97



1.31
technical_bug
0.82
0.22


In [33]:
# 対話全体を踏まえて正確な判断を下せるかを確認

# 1. **検証目的:** 顧客の怒りや不満の度合いを、会話の最後まで追跡した上で正しくスコアリングできるか。

chat_history_1 = """
[User]: 届いた商品が破損しているのですが！どういう管理してるんですか！？即返金してください！
[System]: 大変ご迷惑をおかけし申し訳ございません。直ちに状況を確認の上、返金または交換の手配をさせていただきます。お手数ですが注文番号をお教えいただけますでしょうか。
[User]: 注文番号は #12345 です。セール品だからって適当に梱包しないでほしいです。
[System]: 注文番号のご連絡ありがとうございます。本日中に返金手続き（キャンセル処理）を完了し、完了メールをお送りいたします。この度はお手数とおかけし重ねてお詫び申し上げます。
[User]: 了解です。迅速な対応ありがとうございます。返金メールを待ちます。
"""

response = client.system_one(
    state=chat_history_1,
    questions={
        "final_frustration": Score(
            instructions="会話終了時点における顧客の怒り・不満の度合い",
            criteria=[
                "満足・納得している（問題解決を受け入れた）",
                "まだやや不満・警戒している（渋々納得）",
                "依然として非常に怒っている",
            ],
        ),
    },
)

print(response.answers["final_frustration"].score)  # 0.05〜0.07

# 2. **検証目的:** 顧客の問い合わせ内容が複雑であっても、最優先で対応すべき事項を正しく抽出できるか。

chat_history_2 = """
[User]: こんにちは。有料プランの解約方法を教えてください。
[System]: ご解約の検討ですね。設定画面の「契約管理」→「プラン変更」よりいつでも手続き可能です。
[User]: わかりました。あ、それと先月分の請求書が届いていないのですが再発行できますか？
[System]: 請求書につきましては「請求履歴」タブからPDFでダウンロード可能です。
[User]: 助かります！あ、そういえばログイン用のパスワードを忘れてしまって、別端末から入れなくなりました。リセットメールが届かないのですがどうすればいいですか？
"""

response = client.system_one(
    state=chat_history_2,
    questions={
        "current_needed_action": Choice(
            instructions="対話の最後を踏まえ、現在システムまたはサポートが最優先で対応すべき内容",
            criteria={
                "cancellation": "解約手続きの案内",
                "invoice_resend": "請求書の再発行・送付",
                "password_reset": "ログイン不能・パスワードリセットの不具合対応",
            },
        ),
    },
)

print(response.answers["current_needed_action"].choice)  # "password_reset"

# 3. **検証目的:** 顧客が直接的に「解約」や「不満」を表明していなくても、潜在的なリスクやフォローアップの必要性を正しく検知できるか。

chat_history_3 = """
[User]: APIのレートリミットの上限を上げることはできますか？
[System]: エンタープライズプランへアップグレードいただくか、個別での申請が必要です。ご希望の場合は担当営業からご連絡いたします。
[User]: 営業さんから連絡をもらえると助かります。明日の午後なら電話に出られます。
"""

response = client.system_one(
    state=chat_history_3,
    questions={
        "needs_human_followup": Noul(
            instructions="人間（営業・サポート担当者）による後日のフォローアップやアクションが必要か",
        ),
        "is_resolved": Noul(
            instructions="この対話のみでユーザーの問題や疑問が完全に解決・完結したか",
        ),
    },
)

print(response.answers["needs_human_followup"].noul)  # 0.97
print(response.answers["is_resolved"].noul)           # 0.03

0.03
password_reset
0.94
0.12


In [34]:
# ユーザープロンプトの構造やメタ情報（形式的な特徴・要素の有無）を正確に検知・分類できるかを検証

sample_prompt = """
あなたは10年以上のキャリアを持つベテランのテックライターです。
以下のWebページの内容を参考にして、初心者向けに解説記事を作成してください。

参照URL: https://example.com/tech-article/123

【出力条件】
・Markdown形式の表（Table）を使って比較を含めてください。
・文字数は800文字〜1000文字程度。
・「結論」を最初に記述すること。

絶対に英語は使わず、すべて日本語で出力してください。
"""

response = client.system_one(
    state=sample_prompt,
    questions={
        # 1. URL引用の有無 (Noul)
        "has_url": Noul(
            instructions="テキスト内にWebページのURL（http:// または https://）が含まれているか",
        ),
        
        # 2. 出力形式・フォーマットの指定有無 (Noul)
        "has_format_spec": Noul(
            instructions="Markdownテーブル、JSON、文字数制限、章立てなど、具体的な出力形式やレイアウトに関する指定が含まれているか",
        ),

        # 3. ペルソナ・役割指定の有無 (Noul)
        "has_persona_spec": Noul(
            instructions="「あなたは〇〇です」「〇〇として振る舞ってください」のような役割やペルソナの指定が含まれているか",
        ),

        # 4. 制約条件（ネガティブプロンプト）の有無 (Noul)
        "has_negative_constraint": Noul(
            instructions="「絶対に〇〇するな」「〇〇は除外して」のような禁止事項やネガティブ制約が含まれているか",
        ),

        # 5. プロンプトの主目的分類 (Choice)
        "prompt_intent_type": Choice(
            instructions="このプロンプトの主な命令形式・カテゴリ",
            criteria={
                "summarization_or_rewrite": "文章の要約・翻訳・書き換え・解説依頼",
                "content_generation": "ゼロからの新規コンテンツ作成・アイデア出し",
                "code_or_technical": "プログラミングコードの生成や技術的デバッグ",
                "conversational": "日常会話や単純な質問・雑談",
            },
        ),

        # 6. 指示の複雑さ・制約密度のスコア化 (Score)
        "instruction_complexity": Score(
            instructions="プロンプトに含まれる指示や条件の複雑さ・多さ",
            criteria=[
                "単純（単一の質問や短文指示のみ）",
                "中程度（1〜2個の条件やコンテキスト指定あり）",
                "高密度・複雑（多数の条件、フォーマット指定、禁止事項などが細かく規定されている）",
            ],
        ),
    },
)

# 結果の表示
print("URLの存在:", response.answers["has_url"].noul)                      # 期待値: 0.95〜1.0
print("フォーマット指定:", response.answers["has_format_spec"].noul)        # 期待値: 0.95〜1.0
print("ペルソナ指定:", response.answers["has_persona_spec"].noul)          # 期待値: 0.95〜1.0
print("禁止事項の指定:", response.answers["has_negative_constraint"].noul)  # 期待値: 0.95〜1.0
print("主目的:", response.answers["prompt_intent_type"].choice)           # 期待値: "summarization_or_rewrite"
print("複雑さスコア:", response.answers["instruction_complexity"].score)  # 期待値: 1.8〜2.0 付近

URLの存在: 0.91
フォーマット指定: 0.98
ペルソナ指定: 0.98
禁止事項の指定: 0.98
主目的: content_generation
複雑さスコア: 2.0


In [35]:
import os
from typesafe_sdk import Choice, Noul, Score, TypeSafeClient

client = TypeSafeClient()

# テスト用プロンプト（ビジネス＋ハードウェア/化学系の複雑なテクニカルターム混在）
user_prompt = """
現在、全固体電池の固体電解質（硫化物系）における界面抵抗の低減が課題となっています。
いきなり特定の材料データベースを検索する前に、まずはどのようなアプローチ（被覆処理、ドープ技術、焼結プロセス改質など）が
技術カテゴリとして分類・体系化されているのか、全体的な技術マップの俯瞰とカテゴリの絞り込みを行いたいです。
"""

# ==========================================
# Stage 1: フェーズ（大分類）の判定
# ==========================================
stage1_response = client.system_one(
    state=user_prompt,
    questions={
        "phase": Choice(
            instructions="プロンプトが位置するイノベーション支援プロセスの主要フェーズ",
            criteria={
                "requirements": "課題定義、技術要件の策定、課題のブレイクダウン",
                "tech_search": "技術情報の探索、データベース検索、カテゴリの絞り込み、要素技術の同定",
                "use_case": "技術の用途探索、新市場の開拓、応用アイデアの発想",
                "benchmark": "海外スタートアップ、先進事例、競合他社の動向調査",
                "business_eval": "SWOT分析、事業化評価、特許ポートフォリオ分析",
            },
        )
    },
)

selected_phase = stage1_response.answers["phase"].choice
print(f"[Stage 1 判定結果]: {selected_phase}")

# ==========================================
# Stage 2: 選択されたフェーズに応じた詳細サブ機能の判定
# ==========================================
# ※実際の実装では selected_phase に応じて分岐させます
if selected_phase == "tech_search":
    stage2_response = client.system_one(
        state=user_prompt,
        questions={
            "sub_function": Choice(
                instructions="技術探索フェーズにおける具体的なサブ機能",
                criteria={
                    "category_narrowing": "具体的な技術検索に入る前の、技術カテゴリやドメイン・アプローチの絞り込み・体系化",
                    "deep_db_search": "特定された技術課題や材料・手法に直接アプローチするためのデータベース/論文検索",
                    "supplier_matching": "該当技術を保有する企業・メーカー・研究機関の直接リストアップ",
                    "readiness_eval": "技術の成熟度（TRLレベル）や導入ハードルの判定",
                },
            ),
            "contains_technical_terms": Noul(
                instructions="テキスト内に専門的な技術用語（材料名、物理・化学現象、工学用語など）が明確に含まれているか",
            ),
            "readiness_score": Score(
                instructions="具体的な検索条件がどの程度固まっているか（抽象的な探索〜具体的な条件指定）",
                criteria=[
                    "極めて抽象的（何から探せばいいかわからない）",
                    "方向性は見えているが、カテゴリや軸の整理が必要",
                    "具体的な検索条件・スペック・パラメータが確定している",
                ],
            ),
        },
    )

    print(f"[Stage 2 サブ機能]: {stage2_response.answers['sub_function'].choice}")
    print(f"[テクニカルタームの検出]: {stage2_response.answers['contains_technical_terms'].noul}")
    print(f"[条件の具体化スコア]: {stage2_response.answers['readiness_score'].score}")

[Stage 1 判定結果]: tech_search
[Stage 2 サブ機能]: category_narrowing
[テクニカルタームの検出]: 0.98
[条件の具体化スコア]: 1.0


In [36]:
# Cell 1: モジュールのインポートとアナライザーの初期化
import json
import importlib
import prompt_analyzer
from prompt_analyzer import PromptAnalyzer

# モジュールの変更を強制的反映
importlib.reload(prompt_analyzer)

analyzer = PromptAnalyzer()

# Cell 2: テストプロンプト群の定義
sample_prompts = [
    # パターン1: URLあり ＋ 実効評価（単なる要約ではなくメリット/デメリット評価）
    """
    以下の記事で紹介されているペロブスカイト太陽電池の封止技術について確認してください。
    https://qiita.com/rairaii/items/8673b117096eb0e7e267

    自社の車載向けデバイスに転用する場合の「耐久性の課題」と「製造コスト面でのデメリット」を中心に検証し、
    導入の実現可能性（Feasibility）を評価した比較表をMarkdownで作成してください。
    ※推測による曖昧な情報は記載しないでください。
    """,

    # パターン2: ダミーURLあり ＋ 純粋な要約（後続検討なし）
    """
    以下のURLの論文要約を読んで、主要な主張と結論だけを日本語で簡潔にまとめてください。
    URL: https://arxiv.org/abs/2301.00000

    箇条書きで3点以内に収めて出力してください。評価や分析は不要です。
    """,

    # パターン3: URLなし ＋ 高度課題の技術カテゴリ絞り込み
    """
    現在、全固体電池の固体電解質（硫化物系）における界面抵抗の低減が課題となっています。
    いきなり特定の材料データベースを検索する前に、どのようなアプローチ（被覆処理、ドープ技術、焼結プロセス改質など）が
    技術カテゴリとして分類・体系化されているのか、全体的な技術マップの不確実性を減らすためのカテゴリ絞り込みを行いたいです。
    """,

    # パターン4: URLなし ＋ 仕様・要件定義
    """
    社内のエッジAIカメラ（Jetson Orin搭載）でリアルタイム欠陥検知システムを構築するための技術要件を定義したいです。
    推論速度（FPS）、許容遅延（latency）、画素数、設置環境（防塵防水IP67）の観点から、
    システム開発にあたって考慮すべき仕様・制約条件を漏れなくリストアップしてください。
    必ずJSONフォーマットで出力すること。
    """
]

# Cell 3: 一括テスト実行と結果出力
for idx, prompt in enumerate(sample_prompts, 1):
    print(f"\n==================== TEST CASE {idx} ====================")
    result = analyzer.analyze(prompt)
    
    # 判定結果の見やすい表示
    print(f"URL検出 (Python): {result['url_detected']} -> {result['urls']}")
    print(f"Web取得成功: {result['web_context_fetched']}")
    print(f"判定された目的 (primary_goal): {result['primary_goal']}")
    print(f"メタ構造: {json.dumps(result['meta_structure'], ensure_ascii=False)}")
    print(f"技術用語密度 (0-2): {result['technical_density']}")


==================== TEST CASE 1 ====================
🔗 URLを検出しました: https://qiita.com/rairaii/items/8673b117096eb0e7e267 (Web情報を取得中...)
URL検出 (Python): True -> ['https://qiita.com/rairaii/items/8673b117096eb0e7e267']
Web取得成功: True
判定された目的 (primary_goal): feasibility_eval
メタ構造: {"has_format_spec": 0.88, "has_negative_constraint": 0.84}
技術用語密度 (0-2): 1.05

==================== TEST CASE 2 ====================
🔗 URLを検出しました: https://arxiv.org/abs/2301.00000 (Web情報を取得中...)
URL検出 (Python): True -> ['https://arxiv.org/abs/2301.00000']
Web取得成功: False
判定された目的 (primary_goal): simple_summarization
メタ構造: {"has_format_spec": 0.73, "has_negative_constraint": 0.71}
技術用語密度 (0-2): 0.21

==================== TEST CASE 3 ====================
URL検出 (Python): False -> []
Web取得成功: False
判定された目的 (primary_goal): tech_discovery
メタ構造: {"has_format_spec": 0.07, "has_negative_constraint": 0.25}
技術用語密度 (0-2): 1.95

==================== TEST CASE 4 ====================
URL検出 (Python): False -> []
Web取得成功: False
判定